In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import re

In [ ]:
base_dir = Path(r"C:\Users\myles\OneDrive\Desktop\adhesive_0_0_data")  # contains data_0_0_phi*
folders = sorted(base_dir.glob("data_0_0_phi*"))

In [ ]:
"""Functions find the start and end of the data table in each data file"""


def find_table_start(path):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Time"):
                return i
    raise ValueError(f"No table header found in {path}")


def find_table_end(path):
    with open(path, "r") as f:
        for i, line in enumerate(f):
            if line.strip().startswith("Loop"):
                return i
    return None

In [ ]:
columns = []
column_names = []

for folder in folders:
    # phi from folder name: data_0_0_phi0.55
    mphi = re.search(r"phi([0-9.]+)", folder.name)
    if not mphi:
        continue
    phi_str = mphi.group(1)

    run_files = sorted(folder.glob("run_*"))  # e.g. run_0.01.txt etc.
    print(folder.name, "->", len(run_files), "run files") # checks how many run files are associated with each parent folder

    for file in run_files:
        start = find_table_start(file)
        end = find_table_end(file)

        df = pd.read_csv(
            file,
            delim_whitespace=True,
            skiprows=start,
            nrows=(end - start) if end is not None else None,
            on_bad_lines="skip",
            engine="python"
        )

        col = df.iloc[:, 4]  # 5th column (stress_xy)
        
    

        # shorten filename: run_0.006.txt -> 0.006
        a_str = file.stem.replace("run_", "")



        new_name = f"phi{phi_str}_a{a_str}_stress_xy"
        col = col.rename(new_name)

        columns.append(col)
        column_names.append(new_name)

# combine columns into one dataframe
new_df = pd.concat(columns, axis=1)
new_df.columns = column_names
new_df.insert(0, "Time", range(len(new_df)))

print("Final dataframe shape:", new_df.shape)


new_df.head()

In [ ]:
"""Plot the reduced viscosity against total strain for each simulation"""

plt.figure(figsize=(12, 8))
for i, col in enumerate(new_df.columns[1:]):  # skip Time column
    if i % 14 == 0 and i > 0:
        plt.xlabel("Time")
        plt.ylabel("Reduced Viscosity")
        plt.yscale("log")
        plt.legend()
        plt.show()
        plt.figure(figsize=(12, 8))
    plt.plot(new_df["Time"], np.abs(new_df[col]/ (0.01*0.1)), label=col)
plt.xlabel("Time")
plt.yscale("log")
plt.ylabel("Reduced Viscosity")
plt.title("Reduced Viscosity vs Time for Different phi and a")
plt.legend()
plt.show()

In [ ]:
eta_f = 0.1      # fluid viscosity
gamma_dot = 0.01 # shear rate

# steady state stress is between time 100 and the end
steady_state_stresses = []
for col in new_df.columns[1:]:  # skip Time column
    steady_state_stress = np.abs(new_df[col][100:].mean())  # average stress from time 100 to end
    steady_state_stresses.append(steady_state_stress)

print("Steady state stresses:", steady_state_stresses)



# reduced viscosity = (steady state stress) / (eta_f * gamma_dot)
reduced_viscosities = [stress / (eta_f * gamma_dot) for stress in steady_state_stresses]
print("Reduced viscosities:", reduced_viscosities)

# sem of reduced viscosity = sem of stress / (eta_f * gamma_dot)
sem = []
for col in new_df.columns[1:]:
    sem_value = np.abs(new_df[col][100:].sem())/(eta_f * gamma_dot)  # SEM of stress from time 100 to end
    sem.append(sem_value)

print("SEM of reduced viscosities:", sem)

In [ ]:
# Creating a clear dataframe with columns: phi, a, reduced_viscosity
data = []
for col, eta_r in zip(new_df.columns[1:], reduced_viscosities):
    m = re.match(r"phi([0-9.]+)_a([0-9.]+)_stress_xy", col)
    if m:
        phi = float(m.group(1))
        a = float(m.group(2))
        data.append({"phi": phi, "a": a, "reduced_viscosity": eta_r})
df_viscosity = pd.DataFrame(data)
print(df_viscosity)


In [ ]:
# build style for each phi
phis = sorted({float(re.search(r"phi([0-9.]+)", c).group(1))
               for c in new_df.columns[1:]})
markers = ['o','s','v','^','D','p','*','h','x','+']        
colors  = plt.cm.tab10(np.linspace(0, 1, len(phis)))     

phi_styles = {phi: {'marker': markers[i % len(markers)],
                    'color': colors[i]}
              for i, phi in enumerate(phis)}

# make one figure per packing fraction
for phi in phis:
    plt.figure(figsize=(10, 6))
    for idx, col in enumerate(new_df.columns[1:]):           # skip Time
        m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
        if not m:
            continue
        if float(m.group(1)) != phi:                        # only this phi
            continue
        a = float(m.group(2))

        sigma   = steady_state_stresses[idx]
        sigma_0 = 2 * np.pi * a

        style = phi_styles[phi]
        plt.scatter(sigma / sigma_0,
                    reduced_viscosities[idx],
                    color=style['color'],
                    marker=style['marker'],
                    label=f"a={a}")

    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Sigma / Sigma_0")
    plt.ylabel("Reduced Viscosity")
    plt.title(f"Flow Curve for phi={phi}")
    plt.legend()
    plt.show()
plt.show()

In [ ]:
"""Method to estimate uncertainties on time average steady-state"""
# Fixed correlation time = 1 strain unit
# tau_corr = 1 strain unit = 1/gamma_dot timesteps = 100 timesteps
# N_eff = N / (2 * tau_steps + 1)
# EM_corrected = std / sqrt(N_eff)

tau_steps = 1.0 / gamma_dot          # = 100 timesteps

sem_strain = []
for col in new_df.columns[1:]:       # skip Time
    series = new_df[col][200:].dropna()
    N      = len(series)
    n_eff  = max(N / (2 * tau_steps + 1), 1.0)
    std    = series.std(ddof=1)
    sem_strain.append((std / np.sqrt(n_eff)) / (eta_f * gamma_dot))

sem_strain = np.array(sem_strain)
print("SEM (strain-corrected):", sem_strain)
print(f"Correction factor vs naive: {(sem_strain / np.array([new_df[col][200:].sem() / (eta_f * gamma_dot) for col in new_df.columns[1:]])).mean():.1f}x on average")

In [ ]:
"""Plots reduced viscosity flow curves"""

# Ensures error bars are symmetric on log graph
def log_errorbars(means, sems):
    """Asymmetric log-space error bars so caps are visible on log axes."""
    means = np.asarray(means, dtype=float)
    sems  = np.asarray(sems,  dtype=float)
    rel   = sems / means
    return means * (1 - np.exp(-rel)), means * (np.exp(rel) - 1)

fig, ax = plt.subplots(figsize=(8, 6))
plotted_phis = set()

for idx, col in enumerate(new_df.columns[1:]):
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue
    phi = float(m.group(1))
    a   = float(m.group(2))

    sigma   = steady_state_stresses[idx]
    sigma_0 = 2 * np.pi * a # Adehsion force scale
    eta_r   = reduced_viscosities[idx]

    lo, hi = log_errorbars([eta_r], [sem_strain[idx]])

    style = phi_styles[phi]
    label = f"{phi}" if phi not in plotted_phis else None

    ax.errorbar(sigma / sigma_0, eta_r,
                yerr=[[lo[0]], [hi[0]]],
                fmt=style['marker'],
                color=style['color'],
                capsize=3, capthick=1.2, elinewidth=1.0,
                label=label)

    plotted_phis.add(phi)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\sigma / \sigma_a$", fontsize=22, fontweight='bold')
ax.set_ylabel(r"$\eta_r$", fontsize=22, fontweight='bold')
ax.minorticks_on()
ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax.tick_params(labelsize=17)
ax.legend(fontsize=13, frameon=False, handlelength=0.5, handletextpad=0.3)
plt.tight_layout()
plt.savefig("flowcurve_strain_uncertainty.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Herschel-Bulkley fit
# Model: σ = σ_y + K * γ̇^n


def herschel_bulkley(gamma_dot, sigma_y, K, n):
    return sigma_y + K * np.power(gamma_dot, n)


hb_results = {}  

for phi in phis:
    # Gather all (a, sigma) points for this phi
    a_vals, sigma_vals = [], []
    for idx, col in enumerate(new_df.columns[1:]):
        m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
        if not m:
            continue
        if float(m.group(1)) != phi:
            continue
        a_vals.append(float(m.group(2)))
        sigma_vals.append(steady_state_stresses[idx])

    a_arr     = np.array(a_vals)
    sigma_arr = np.array(sigma_vals)
    sort_idx  = np.argsort(a_arr)
    a_arr     = a_arr[sort_idx]
    sigma_arr = sigma_arr[sort_idx]

    try:
        # Initial guess: sigma_y ~ min stress, K ~ 1, n ~ 0.5
        p0     = [sigma_arr.min() * 0.5, 1.0, 0.5]
        bounds = ([0, 0, 0], [np.inf, np.inf, 2.0])   # sigma_y, K >= 0; n in [0,2]
        popt, pcov = curve_fit(
            herschel_bulkley, a_arr, sigma_arr,
            p0=p0, bounds=bounds, maxfev=10000
        )
        perr = np.sqrt(np.diag(pcov))   # 1-sigma parameter uncertainties
        hb_results[phi] = {
            'a':        a_arr,
            'sigma':    sigma_arr,
            'sigma_y':  popt[0], 'sigma_y_err': perr[0],
            'K':        popt[1], 'K_err':       perr[1],
            'n':        popt[2], 'n_err':       perr[2],
        }
        print(f"phi={phi:.4f}  σ_y={popt[0]:.4f}±{perr[0]:.4f}  "
              f"K={popt[1]:.4f}±{perr[1]:.4f}  n={popt[2]:.4f}±{perr[2]:.4f}")
    except RuntimeError as e:
        print(f"phi={phi}: fit failed — {e}")

# Plot: data + HB fits 
fig, ax = plt.subplots(figsize=(10, 6))

for phi, res in hb_results.items():
    style = phi_styles[phi]
    ax.scatter(res['a'], res['sigma'],
               color=style['color'], marker=style['marker'],
               label=f"phi={phi}", zorder=3)

    # Smooth fit line
    a_fine     = np.linspace(res['a'].min(), res['a'].max(), 300)
    sigma_fine = herschel_bulkley(a_fine,
                                  res['sigma_y'], res['K'], res['n'])
    ax.plot(a_fine, sigma_fine, color=style['color'], lw=1.5, ls='--')

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel(r"$\dot{\gamma}$ (shear rate $a$)", fontsize=14, fontweight='bold')
ax.set_ylabel(r"Steady-state stress $\sigma$",    fontsize=14, fontweight='bold')
ax.set_title("Herschel-Bulkley fits", fontsize=14)
ax.minorticks_on()
ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)
ax.legend()
plt.tight_layout()
plt.show()

# ── Yield stress vs phi summary ───────────────────────────────────────────────
phi_list    = sorted(hb_results.keys())
sigma_y_arr = np.array([hb_results[p]['sigma_y']     for p in phi_list])
sigma_y_err = np.array([hb_results[p]['sigma_y_err'] for p in phi_list])

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(phi_list, sigma_y_arr, yerr=sigma_y_err,
            fmt='o-', capsize=4, color='steelblue')
ax.set_xlabel(r"Packing fraction $\phi$", fontsize=14, fontweight='bold')
ax.set_ylabel(r"Yield stress $\sigma_y$",  fontsize=14, fontweight='bold')
ax.set_title("Yield stress vs packing fraction", fontsize=14)
ax.minorticks_on()
ax.tick_params(axis='both', which='both', direction='in', top=True, right=True)
plt.tight_layout()
plt.show()

In [ ]:
"""Plot graph used to determine yeild stress"""

from scipy.optimize import curve_fit

for idx, col in enumerate(new_df.columns[1:]):           # skip Time
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue
    phi = float(m.group(1))
    a   = float(m.group(2))

    sigma   = steady_state_stresses[idx] 
    sigma_0 = 2 * np.pi * a

    style = phi_styles[phi]
    label = f"phi={phi}" if phi not in plotted_phis else None


    plt.scatter((3*0.1*0.01)/a,
                3*sigma/a,
                color=style['color'],
                marker=style['marker'],
                label=label)

    plotted_phis.add(phi)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("1 / Sigma_0")
plt.ylabel("Stress")
plt.title("Flow Curves: Stress vs Sigma/Sigma_0")
plt.legend()
plt.show()

In [ ]:
# --- 1) collect points per phi ---
from collections import defaultdict


points_by_phi = defaultdict(list)

for idx, col in enumerate(new_df.columns[1:]):  # skip Time
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue

    phi = float(m.group(1))
    a   = float(m.group(2))

    sigma   = steady_state_stresses[idx]    #  y value (stress)
    sigma_0 = 2 * np.pi * a                #  stress scale
    x = 1/sigma_0           

    points_by_phi[phi].append((x, sigma))

# --- 2) fit first 3 smallest-x points for each phi, print intercept ---
sigma_y_by_phi = {}

for phi, pts in sorted(points_by_phi.items()):
    pts_sorted = sorted(pts, key=lambda t: t[0])   # sort by x (ascending)
    if len(pts_sorted) < 3:
        print(f"phi={phi}: only {len(pts_sorted)} points -> skipping")
        continue

    xs = np.array([p[0] for p in pts_sorted[:3]])
    ys = np.array([p[1] for p in pts_sorted[:3]])

    # linear fit: y = m x + b
    m_fit, b_fit = np.polyfit(xs, ys, 1)
    sigma_y_by_phi[phi] = b_fit

    print(f"phi={phi}: fit on smallest 3 x -> sigma_y (y-intercept) = {b_fit:.6g}, slope = {m_fit:.6g}")

In [ ]:
points_by_phi = defaultdict(list)

# collect data 
for idx, col in enumerate(new_df.columns[1:]):  # skip Time
    m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
    if not m:
        continue

    phi = float(m.group(1))
    a   = float(m.group(2))

    sigma   = steady_state_stresses[idx] *(3/a)
    sigma_0 = 2 * np.pi * a
    x = (3*0.1*0.01)/a

    points_by_phi[phi].append((x, sigma))

#  plot + fit
plotted_phis = set()  
for phi, pts in sorted(points_by_phi.items()):

    style = phi_styles[phi]
    label = f"{phi}" if phi not in plotted_phis else None

    # sort by smallest x
    pts_sorted = sorted(pts, key=lambda t: t[0])

    xs = np.array([p[0] for p in pts_sorted])
    ys = np.array([p[1] for p in pts_sorted])

    # scatter all points
    plt.scatter(xs,
                ys,
                color=style['color'],
                marker=style['marker'],
                label=label)

    plotted_phis.add(phi)

    # Fit first 3 smallest x points 
    if len(xs) >= 3:
        xs_fit = xs[:3]
        ys_fit = ys[:3]

        # linear model
        def model(x, m, b):
            return m * x + b

        popt, _ = curve_fit(model, xs_fit, ys_fit)

        m_fit, b_fit = popt

        print(f"phi={phi}: sigma_y (y-intercept) = {b_fit:.6g}")

        # plot fitted line over full x-range
        x_line = np.linspace(0, xs_fit[-1], 200)
        y_line = model(x_line, m_fit, b_fit)

        plt.plot(x_line,
                 y_line,
                 color=style['color'],
                 linestyle='--'
                 )

plt.xscale("log")
plt.yscale("log")
plt.xlabel("$\hat{\dot\gamma}$", fontsize=16, fontweight='bold')
plt.ylabel("$\hat{\sigma}$", fontsize=16, fontweight='bold')
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(fontsize=11, frameon=False, handlelength=0.5, handletextpad=0.3)
plt.savefig("yield_stress_determine.png", dpi= 1200, bbox_inches="tight")
plt.show()

In [ ]:
# Plot the yield stress (sigma_y) vs phi using the fitted intercepts
plt.figure(figsize=(8, 6))
phis = sorted(sigma_y_by_phi.keys())
sigma_ys = [sigma_y_by_phi[phi] for phi in phis]
plt.plot(phis, sigma_ys, marker='o', linestyle='')
plt.xlabel("Phi")
plt.yscale("log")
plt.ylabel("Yield Stress (sigma_y)")
plt.title("Yield Stress vs Packing Fraction")
plt.show()

df_export = pd.DataFrame({
    "phi" : phis,
    "sigma_y" : sigma_ys
})

df_export.to_csv("0_0_flowcurve.csv", index=False)

In [ ]:
"""Extract the coordination number from each file"""

columns_1 = []
column_1_names = []

for folder in folders:
    # phi from folder name: data_0_0_phi0.55
    mphi = re.search(r"phi([0-9.]+)", folder.name)
    if not mphi:
        continue
    phi_str = mphi.group(1)

    run_files = sorted(folder.glob("run_*"))  # e.g. run_0.01.txt etc.
    print(folder.name, "->", len(run_files), "run files")

    for file in run_files:
        start = find_table_start(file)
        end = find_table_end(file)

        df2 = pd.read_csv(
            file,
            delim_whitespace=True,
            skiprows=start,
            nrows=(end - start) if end is not None else None,
            on_bad_lines="skip",
            engine="python"
        )

        col = df2.iloc[:, 7]  # 8th column (contact_number)
        
    

        # a from filename safely: run_0.006.txt -> 0.006
        a_str = file.stem.replace("run_", "")



        new_name = f"phi{phi_str}_a{a_str}_contact_number"
        col = col.rename(new_name)

        columns_1.append(col)
        column_1_names.append(new_name)

# combine columns into data frame
new_df2 = pd.concat(columns_1, axis=1)
new_df2.columns = column_1_names
new_df2.insert(0, "Time", range(len(new_df2)))

print("Final dataframe shape:", new_df2.shape)


print(new_df2)

In [ ]:
steady = new_df2[(new_df2["Time"] >= 100) & (new_df2["Time"] <= 3000)]

# mean for every column except Time
steady_means = steady.drop(columns=["Time"]).mean()

print(steady_means)




In [ ]:
records = []
for name, zbar in steady_means.items():
    m = re.search(r"phi([0-9.]+)_a([0-9.eE+-]+)_contact_number", name)
    if not m:
        continue
    records.append({"phi": float(m.group(1)), "a": float(m.group(2)), "Z_mean": zbar})

tidy = pd.DataFrame(records).sort_values(["a", "phi"])

import pickle
plt.figure(figsize=(10, 4))

for phi in phis:
    style = phi_styles[phi]

    xs = []
    zs = []
    for idx, col in enumerate(new_df.columns[1:]):   # skip Time
        m = re.search(r"phi([0-9.]+)_a([0-9.]+)", col)
        if not m:
            continue
        if float(m.group(1)) != phi:                # only this φ
            continue
        a = float(m.group(2))

        sigma   = steady_state_stresses[idx]
        sigma_0 = 2 * np.pi * a

        z_mean = tidy[(tidy["phi"] == phi) & (tidy["a"] == a)]["Z_mean"].values[0]
        xs.append(sigma / sigma_0)
        zs.append(z_mean)

    # sort by x so the connecting line is monotonic
    order = np.argsort(xs)
    xs = np.array(xs)[order]
    zs = np.array(zs)[order]

    plt.plot(xs, zs,
             marker=style['marker'],
             color=style['color'],
             linestyle='-',
             label=f"φ={phi}")

plt.xscale("log")
plt.xlabel("$\sigma / \sigma_0$", fontsize=14, fontweight='bold')
plt.ylabel("⟨Z⟩", fontsize=14, fontweight='bold')
plt.yticks(size=14)
plt.xticks(size=14)
plt.legend(fontsize=12)
plt.tight_layout()
fig = plt.gcf()  # grab the current figure
with open('fig1.pkl', 'wb') as f:  # change to fig2.pkl, fig3.pkl
    pickle.dump(fig, f)
plt.show()